In [3]:
import sys
import qprep

print(sys.executable)
print(qprep.__file__)

d:\FinalYearProject\quantum-state-preparation\.venv\Scripts\python.exe
D:\FinalYearProject\quantum-state-preparation\src\qprep\__init__.py


In [4]:
from qprep.smoke_check import main

main()

     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘
Statevector: [0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]
Bell-state fidelity: 1.000000000000


In [5]:
from pathlib import Path

from qprep.config import load_config

# Handle starting the notebook from the project root or notebooks/.
project_root = Path.cwd().resolve()

if not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent

if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("Open this notebook from the project workspace.")

config = load_config(project_root / "configs" / "demo1.json")

print("Experiment:", config["experiment_name"])
print("Observation dimension:", config["observation_dim"])
print("Number of actions:", config["n_actions"])

Experiment: demo1
Observation dimension: 17
Number of actions: 8


In [6]:
from qprep.runtime import prepare_run

run = prepare_run(
    config,
    seed=config["seeds"][0],
    project_root=project_root,
    device="cpu",
)

print("Run directory:", run.run_dir)
print("Seed:", run.seed)
print("Device:", run.device)
print("Random streams:", list(run.rngs))

Run directory: D:\FinalYearProject\quantum-state-preparation\runs\20261111T041600614321Z_seed0_53ae9c46
Seed: 0
Device: cpu
Random streams: ['targets', 'exploration', 'replay', 'baseline', 'evaluation']


In [7]:
from qprep.actions import build_action_catalogue, get_action

for action in build_action_catalogue():
    print(
        f"ID {action.action_id}: "
        f"{action.gate.upper()} on qubits {action.qubits}"
    )

print("\nExample lookup:", get_action(6))

ID 0: H on qubits (0,)
ID 1: H on qubits (1,)
ID 2: S on qubits (0,)
ID 3: S on qubits (1,)
ID 4: T on qubits (0,)
ID 5: T on qubits (1,)
ID 6: CX on qubits (0, 1)
ID 7: CX on qubits (1, 0)

Example lookup: Action(action_id=6, gate='cx', qubits=(0, 1))


In [8]:
import numpy as np

from qprep.quantum import validate_state

bell_state = validate_state(
    [2**-0.5, 0, 0, 2**-0.5],
    n=2,
)

print("State:", bell_state)
print("Dtype:", bell_state.dtype)
print("Squared norm:", np.vdot(bell_state, bell_state).real)

try:
    validate_state([1, 0, 0, 1], n=2)
except ValueError as error:
    print("Correctly rejected:", error)

State: [0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]
Dtype: complex128
Squared norm: 1.0000000000000002
Correctly rejected: State must be normalised: squared norm is 2; allowed absolute error is 1e-10.


In [9]:
from qprep.quantum import apply_gate, zero_state

state = zero_state(2)
for action_id in (0, 6):
    state = apply_gate(state, action_id)
    print(f"After action {action_id}: {state}")

After action 0: [ 0.70710678+0.j  0.70710678+0.j  0.        +0.j -0.        +0.j]
After action 6: [0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]


In [10]:
from qprep.baseline import run_random_baseline, verify_and_summarise_baseline
from qprep.runtime import prepare_run, write_json

baseline_run = prepare_run(
    config,
    seed=config["seeds"][0],
    project_root=project_root,
    device="cpu",
)

traces_path = run_random_baseline(baseline_run)
summary = verify_and_summarise_baseline(traces_path, config)
write_json(baseline_run.run_dir / "random_baseline_summary.json", summary)

for target_id, result in summary.items():
    print(
        f"{target_id}: {result['successes']}/{result['attempts']} "
        f"({result['success_rate']:.1%})"
    )

print("Saved to:", baseline_run.run_dir)

A: 43/100 (43.0%)
B: 42/100 (42.0%)
C: 19/100 (19.0%)
D: 12/100 (12.0%)
E: 14/100 (14.0%)
F: 12/100 (12.0%)
Saved to: D:\FinalYearProject\quantum-state-preparation\runs\20261111T041608870670Z_seed0_cb8b4615


In [11]:
from qprep.config import load_config
from qprep.runtime import prepare_run
from qprep.train import (
    Trainer,
    restore_training_checkpoint,
    train_to_next_checkpoint,
)

smoke_config = load_config(project_root / "configs" / "demo1.json")
smoke_config.update({
    "training_steps": 32,
    "warmup_steps": 8,
    "batch_size": 4,
    "replay_capacity": 64,
    "epsilon_decay_steps": 24,
    "target_update_interval": 4,
    "eval_interval": 16,
})

output_root = project_root / "runs" / "smoke"
seed = smoke_config["seeds"][0]

first_run = prepare_run(
    smoke_config,
    seed=seed,
    project_root=project_root,
    output_root=output_root,
    device="cpu",
)
trainer = Trainer(first_run)
first_path, first_evaluation = train_to_next_checkpoint(trainer)

print("First checkpoint:", first_path)
print("Progress at save:", trainer.progress)

First checkpoint: D:\FinalYearProject\quantum-state-preparation\runs\smoke\20261111T041613700080Z_seed0_b3a74003\checkpoint_000016.pt
Progress at save: {'environment_steps': 16, 'gradient_steps': 9, 'episodes_completed': 2, 'epsilon': 0.3666666666666667}


In [12]:
resumed_run = prepare_run(
    smoke_config,
    seed=seed,
    project_root=project_root,
    output_root=output_root,
    device="cpu",
)
resumed = Trainer(resumed_run)
restore_training_checkpoint(resumed, first_path)

print("Restored progress:", resumed.progress)

final_path, final_evaluation = train_to_next_checkpoint(resumed)

print("Final checkpoint:", final_path)
print("Final progress:", resumed.progress)
print("Evaluated targets:", [row["target_id"] for row in final_evaluation])

Restored progress: {'environment_steps': 16, 'gradient_steps': 9, 'episodes_completed': 2, 'epsilon': 0.3666666666666667}
Final checkpoint: D:\FinalYearProject\quantum-state-preparation\runs\smoke\20261111T041629601711Z_seed0_215c7439\checkpoint_000032.pt
Final progress: {'environment_steps': 32, 'gradient_steps': 25, 'episodes_completed': 5, 'epsilon': 0.050000000000000044}
Evaluated targets: ['A', 'B', 'C', 'D', 'E', 'F']


In [ ]:
import json
from pathlib import Path

from qprep.config import load_config
from qprep.runtime import prepare_run, write_json
from qprep.train import (
    Trainer,
    restore_training_checkpoint,
    train_to_next_checkpoint,
)

main_config = load_config(project_root / "configs" / "demo1.json")

experiment_dir = project_root / "runs" / "demo1-main"
experiment_dir.mkdir(parents=True, exist_ok=True)

selection_rule = {
    "rule": "Use the final scheduled checkpoint from each configured seed.",
    "environment_steps": main_config["training_steps"],
    "seeds": main_config["seeds"],
}

rule_path = experiment_dir / "selection_rule.json"
if rule_path.is_file():
    saved_rule = json.loads(rule_path.read_text(encoding="utf-8"))
    if saved_rule != selection_rule:
        raise ValueError("This folder already has a different selection rule.")
else:
    write_json(rule_path, selection_rule)

print(selection_rule)

{'rule': 'Use the final scheduled checkpoint from each configured seed.', 'environment_steps': 100000, 'seeds': [0, 1, 2]}


In [14]:
def train_seed(seed, resume_from=None):
    record_path = experiment_dir / f"seed_{seed}_final.json"
    if record_path.is_file():
        return json.loads(record_path.read_text(encoding="utf-8"))

    run = prepare_run(
        main_config,
        seed=seed,
        project_root=project_root,
        output_root=experiment_dir,
        device="cpu",
    )
    trainer = Trainer(run)
    print(f"Seed {seed} run folder: {run.run_dir}", flush=True)

    if resume_from is not None:
        checkpoint_path = Path(resume_from).resolve()
        saved_seed = restore_training_checkpoint(
            trainer, checkpoint_path
        )["provenance"]["seed"]
        if saved_seed != seed:
            raise ValueError(f"Checkpoint belongs to seed {saved_seed}.")

    while trainer.progress["environment_steps"] < main_config["training_steps"]:
        checkpoint_path, _ = train_to_next_checkpoint(trainer)

        step = trainer.progress["environment_steps"]
        minutes = trainer.training_seconds / 60
        print(
            f"Seed {seed} | {step:,}/{main_config['training_steps']:,} steps "
            f"| {minutes:.1f} min training | {checkpoint_path.name}",
            flush=True,
        )

    record = {
        "seed": seed,
        "environment_steps": trainer.progress["environment_steps"],
        "gradient_steps": trainer.progress["gradient_steps"],
        "training_seconds": trainer.training_seconds,
        "checkpoint": checkpoint_path.relative_to(experiment_dir).as_posix(),
    }
    write_json(record_path, record)
    return record

In [13]:
train_seed(main_config["seeds"][0])

Seed 0 run folder: D:\FinalYearProject\quantum-state-preparation\runs\demo1-main\20261111T032935735842Z_seed0_36010bd6
Seed 0 | 5,000/100,000 steps | 0.2 min training | checkpoint_005000.pt
Seed 0 | 10,000/100,000 steps | 0.5 min training | checkpoint_010000.pt
Seed 0 | 15,000/100,000 steps | 0.8 min training | checkpoint_015000.pt
Seed 0 | 20,000/100,000 steps | 1.0 min training | checkpoint_020000.pt
Seed 0 | 25,000/100,000 steps | 1.3 min training | checkpoint_025000.pt
Seed 0 | 30,000/100,000 steps | 1.6 min training | checkpoint_030000.pt
Seed 0 | 35,000/100,000 steps | 1.9 min training | checkpoint_035000.pt
Seed 0 | 40,000/100,000 steps | 2.1 min training | checkpoint_040000.pt
Seed 0 | 45,000/100,000 steps | 2.4 min training | checkpoint_045000.pt
Seed 0 | 50,000/100,000 steps | 2.7 min training | checkpoint_050000.pt
Seed 0 | 55,000/100,000 steps | 3.0 min training | checkpoint_055000.pt
Seed 0 | 60,000/100,000 steps | 3.2 min training | checkpoint_060000.pt
Seed 0 | 65,000/10

{'seed': 0,
 'environment_steps': 100000,
 'gradient_steps': 99001,
 'training_seconds': 330.111338300223,
 'checkpoint': '20261111T032935735842Z_seed0_36010bd6/checkpoint_100000.pt'}

In [14]:
train_seed(main_config["seeds"][1])

Seed 1 run folder: D:\FinalYearProject\quantum-state-preparation\runs\demo1-main\20261111T033534392714Z_seed1_ddd19986
Seed 1 | 5,000/100,000 steps | 0.3 min training | checkpoint_005000.pt
Seed 1 | 10,000/100,000 steps | 0.7 min training | checkpoint_010000.pt
Seed 1 | 15,000/100,000 steps | 1.3 min training | checkpoint_015000.pt
Seed 1 | 20,000/100,000 steps | 1.7 min training | checkpoint_020000.pt
Seed 1 | 25,000/100,000 steps | 2.1 min training | checkpoint_025000.pt
Seed 1 | 30,000/100,000 steps | 2.3 min training | checkpoint_030000.pt
Seed 1 | 35,000/100,000 steps | 2.6 min training | checkpoint_035000.pt
Seed 1 | 40,000/100,000 steps | 2.9 min training | checkpoint_040000.pt
Seed 1 | 45,000/100,000 steps | 3.1 min training | checkpoint_045000.pt
Seed 1 | 50,000/100,000 steps | 3.4 min training | checkpoint_050000.pt
Seed 1 | 55,000/100,000 steps | 3.7 min training | checkpoint_055000.pt
Seed 1 | 60,000/100,000 steps | 4.0 min training | checkpoint_060000.pt
Seed 1 | 65,000/10

{'seed': 1,
 'environment_steps': 100000,
 'gradient_steps': 99001,
 'training_seconds': 588.2748945999774,
 'checkpoint': '20261111T033534392714Z_seed1_ddd19986/checkpoint_100000.pt'}

In [15]:
train_seed(main_config["seeds"][2])

Seed 2 run folder: D:\FinalYearProject\quantum-state-preparation\runs\demo1-main\20261111T034617311886Z_seed2_8bec6f8c
Seed 2 | 5,000/100,000 steps | 0.5 min training | checkpoint_005000.pt
Seed 2 | 10,000/100,000 steps | 1.0 min training | checkpoint_010000.pt
Seed 2 | 15,000/100,000 steps | 1.5 min training | checkpoint_015000.pt
Seed 2 | 20,000/100,000 steps | 2.0 min training | checkpoint_020000.pt
Seed 2 | 25,000/100,000 steps | 2.7 min training | checkpoint_025000.pt
Seed 2 | 30,000/100,000 steps | 3.2 min training | checkpoint_030000.pt
Seed 2 | 35,000/100,000 steps | 3.6 min training | checkpoint_035000.pt
Seed 2 | 40,000/100,000 steps | 4.0 min training | checkpoint_040000.pt
Seed 2 | 45,000/100,000 steps | 4.4 min training | checkpoint_045000.pt
Seed 2 | 50,000/100,000 steps | 4.8 min training | checkpoint_050000.pt
Seed 2 | 55,000/100,000 steps | 5.2 min training | checkpoint_055000.pt
Seed 2 | 60,000/100,000 steps | 5.8 min training | checkpoint_060000.pt
Seed 2 | 65,000/10

{'seed': 2,
 'environment_steps': 100000,
 'gradient_steps': 99001,
 'training_seconds': 587.2244471001904,
 'checkpoint': '20261111T034617311886Z_seed2_8bec6f8c/checkpoint_100000.pt'}

In [15]:
selected = []

for seed in main_config["seeds"]:
    record_path = experiment_dir / f"seed_{seed}_final.json"
    record = json.loads(record_path.read_text(encoding="utf-8"))

    if record["seed"] != seed:
        raise ValueError(f"Final record has the wrong seed for {seed}.")
    if record["environment_steps"] != main_config["training_steps"]:
        raise ValueError(f"Seed {seed} has not reached the final step.")
    if not (experiment_dir / record["checkpoint"]).is_file():
        raise FileNotFoundError(record["checkpoint"])

    selected.append(record)

manifest = {
    "selection_rule": selection_rule,
    "models": selected,
}

manifest_path = experiment_dir / "selected_checkpoints.json"
if manifest_path.is_file():
    saved_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if saved_manifest != manifest:
        raise ValueError("The selected checkpoints have already been frozen.")
else:
    write_json(manifest_path, manifest)

print("Selected checkpoints:", manifest_path)

Selected checkpoints: D:\FinalYearProject\quantum-state-preparation\runs\demo1-main\selected_checkpoints.json


In [16]:
import json
from pathlib import Path

import numpy as np
import torch

from qprep.actions import serialise_action_catalogue
from qprep.dqn import QNetwork
from qprep.evaluate import evaluate_policy

baseline_path = Path(r"D:\FinalYearProject\quantum-state-preparation\runs\20261111T032925816770Z_seed0_50fc82e2\random_baseline_summary.json")
baseline = json.loads(baseline_path.read_text(encoding="utf-8"))

manifest = json.loads(
    (experiment_dir / "selected_checkpoints.json").read_text(encoding="utf-8")
)
models = manifest["models"]
target_ids = main_config["target_ids"]

if [model["seed"] for model in models] != main_config["seeds"]:
    raise ValueError("The manifest must contain one final model per configured seed.")

if set(baseline) != set(target_ids):
    raise ValueError("The baseline and DQN must use the same six targets.")

if any(
    baseline[target_id]["attempts"] != main_config["random_attempts_per_target"]
    for target_id in target_ids
):
    raise ValueError("The baseline is missing attempts.")

rows = []
per_seed = []

for model in models:
    seed = model["seed"]
    checkpoint_path = experiment_dir / model["checkpoint"]
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

    if checkpoint["environment_steps"] != main_config["training_steps"]:
        raise ValueError(f"Seed {seed} is missing its final checkpoint.")
    if checkpoint["provenance"]["seed"] != seed:
        raise ValueError(f"Checkpoint seed does not match manifest seed {seed}.")
    if checkpoint["config"] != main_config:
        raise ValueError(f"Checkpoint configuration differs for seed {seed}.")
    if checkpoint["action_catalogue"] != serialise_action_catalogue():
        raise ValueError(f"Checkpoint action catalogue differs for seed {seed}.")

    network = QNetwork(main_config)
    network.load_state_dict(checkpoint["online_network_state"])

    timed_results = evaluate_policy(
        network, main_config, np.random.default_rng(seed),
        measure_inference=True,
    )

    saved_results = checkpoint["evaluation"]
    if [row["target_id"] for row in saved_results] != target_ids:
        raise ValueError(f"Saved evaluation has the wrong targets for seed {seed}.")

    successes = 0
    for saved, timed in zip(saved_results, timed_results, strict=True):
        inference_seconds = timed.pop("inference_seconds")

        if timed != saved:
            raise ValueError(
                f"Evaluation differs for seed {seed}, target {saved['target_id']}."
            )

        successes += int(saved["success"])
        rows.append({
            "seed": seed,
            "target_id": saved["target_id"],
            "success": saved["success"],
            "final_fidelity": saved["final_fidelity"],
            "gates_used": saved["gate_count"],
            "successful_gate_count": (
                saved["gate_count"] if saved["success"] else None
            ),
            "depth": saved["depth"],
            "cx_count": saved["cx_count"],
            "end_reason": saved["trace"]["end_reason"],
            "inference_ms": inference_seconds * 1000,
            "trace": saved["trace"],
        })

    per_seed.append({
        "seed": seed,
        "successes": successes,
        "attempts": len(target_ids),
        "training_seconds": model["training_seconds"],
    })

In [17]:
dqn_successes = sum(row["success"] for row in rows)
dqn_attempts = len(rows)
random_successes = sum(result["successes"] for result in baseline.values())
random_attempts = sum(result["attempts"] for result in baseline.values())

dqn_rate = dqn_successes / dqn_attempts
random_rate = random_successes / random_attempts

majority_met = 2 * dqn_successes > dqn_attempts
baseline_beaten = dqn_rate > random_rate
learning_gate_met = majority_met and baseline_beaten

results = {
    "dqn": {
        "successes": dqn_successes,
        "attempts": dqn_attempts,
        "success_rate": dqn_rate,
    },
    "random": {
        "successes": random_successes,
        "attempts": random_attempts,
        "success_rate": random_rate,
        "by_target": baseline,
    },
    "per_seed": per_seed,
    "attempts": rows,
    "majority_met": majority_met,
    "baseline_beaten": baseline_beaten,
    "learning_gate_met": learning_gate_met,
}

results_path = experiment_dir / "demo1_results.json"
results_path.write_text(
    json.dumps(results, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

print(f"DQN:    {dqn_successes}/{dqn_attempts} ({dqn_rate:.1%})")
print(f"Random: {random_successes}/{random_attempts} ({random_rate:.1%})")
print("Learning criterion:", "met" if learning_gate_met else "incomplete")

for seed_result in per_seed:
    print(
        f"Seed {seed_result['seed']} | "
        f"{seed_result['successes']}/{seed_result['attempts']} successes | "
        f"{seed_result['training_seconds']:.1f} s training"
    )

print("\nSeed Target Outcome             Fidelity Gates Depth CX  Policy ms")
for row in rows:
    print(
        f"{row['seed']:>4} {row['target_id']:>6} "
        f"{row['end_reason']:<19} "
        f"{row['final_fidelity']:>8.6f} "
        f"{row['gates_used']:>5} {row['depth']:>5} "
        f"{row['cx_count']:>2} {row['inference_ms']:>10.3f}"
    )

print("\nSaved:", results_path)

DQN:    18/18 (100.0%)
Random: 142/600 (23.7%)
Learning criterion: met
Seed 0 | 6/6 successes | 330.1 s training
Seed 1 | 6/6 successes | 588.3 s training
Seed 2 | 6/6 successes | 587.2 s training

Seed Target Outcome             Fidelity Gates Depth CX  Policy ms
   0      A success             1.000000     7     5  2      1.579
   0      B success             1.000000     5     4  0      0.912
   0      C success             1.000000     7     6  2      1.194
   0      D success             1.000000     2     2  0      0.621
   0      E success             1.000000     8     7  1      1.437
   0      F success             1.000000     7     5  1      1.176
   1      A success             1.000000     3     3  0      0.838
   1      B success             1.000000     2     1  0      0.352
   1      C success             1.000000     8     4  0      1.624
   1      D success             1.000000     7     7  2      1.662
   1      E success             1.000000     8     5  1      1.33

In [18]:
import json
from importlib.metadata import version
from pathlib import Path

import numpy as np
import torch

from qprep.actions import serialise_action_catalogue
from qprep.dqn import QNetwork
from qprep.evaluate import evaluate_policy

experiment_dir = Path(r"D:\FinalYearProject\quantum-state-preparation\runs\demo1-main")
manifest = json.loads(
    (experiment_dir / "selected_checkpoints.json").read_text(encoding="utf-8")
)
rule = manifest["selection_rule"]
models = manifest["models"]

if [model["seed"] for model in models] != rule["seeds"]:
    raise ValueError("The manifest must contain one checkpoint per selected seed.")

checks = []

for model in models:
    seed = model["seed"]
    path = experiment_dir / model["checkpoint"]
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    config = checkpoint["config"]

    if checkpoint["schema_version"] != 1:
        raise ValueError(f"Unsupported checkpoint version for seed {seed}.")
    if checkpoint["provenance"]["seed"] != seed:
        raise ValueError(f"Checkpoint seed differs from manifest seed {seed}.")
    if checkpoint["environment_steps"] != rule["environment_steps"]:
        raise ValueError(f"Seed {seed} is missing its final checkpoint.")
    if checkpoint["action_catalogue"] != serialise_action_catalogue():
        raise ValueError(f"Action catalogue differs for seed {seed}.")

    network = QNetwork(config)
    network.load_state_dict(checkpoint["online_network_state"])

    reloaded = evaluate_policy(network, config, np.random.default_rng(seed))
    saved = checkpoint["evaluation"]

    if [row["target_id"] for row in saved] != config["target_ids"]:
        raise ValueError(f"Saved evaluation has the wrong targets for seed {seed}.")

    for expected, actual in zip(saved, reloaded, strict=True):
        if actual != expected:
            changed = [key for key in expected if actual.get(key) != expected[key]]
            raise ValueError(
                f"Seed {seed}, target {expected['target_id']} differs. "
                f"Fields: {changed}"
            )

    successes = sum(row["success"] for row in reloaded)
    checks.append({
        "seed": seed,
        "checkpoint": model["checkpoint"],
        "successes": successes,
        "targets_matched": len(reloaded),
    })
    print(f"Seed {seed}: {len(reloaded)} evaluations matched, {successes} successes")

report = {
    "device": "cpu",
    "packages": {
        name: version(name)
        for name in ("torch", "numpy", "qiskit", "gymnasium")
    },
    "checks": checks,
}

report_path = experiment_dir / "e9_reload_check.json"
report_path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(f"Reload check saved to {report_path}")

Seed 0: 6 evaluations matched, 6 successes
Seed 1: 6 evaluations matched, 6 successes
Seed 2: 6 evaluations matched, 6 successes
Reload check saved to D:\FinalYearProject\quantum-state-preparation\runs\demo1-main\e9_reload_check.json
